# Decoder-Only Transformer (GPT-style) in TensorFlow / Keras 3

Trains a small causal language model on WikiText-103 with the GPT-2 tokenizer.

**Fixes applied vs. the original code**
1. Removed the duplicated `TokenEmbedding` / `PositionalEncoding` / `MultiHeadAttention` / ... definitions and the stray `pe[:, 1::2] = ...` lines that had been pasted into the middle of the `Transformer` class.
2. `dec_layer(x, training, mask)` -> `dec_layer(x, training=training, look_ahead_mask=mask)`. Keras 3 only allows tensors as positional args, which is exactly what the `ValueError` was complaining about.
3. Renamed the `mask` argument to `look_ahead_mask`. `mask` is a reserved keyword in `Layer.__call__` and gets intercepted by Keras' masking machinery.
4. `FeedForward` now actually receives `training`, so its dropout is disabled at inference.
5. Validation data is no longer shuffled; training stream is `.repeat()`ed so `steps_per_epoch` advances through the corpus instead of replaying the same first batches every epoch.

## 1. Setup

In [ ]:
!pip install -q tensorflow datasets transformers numpy matplotlib

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Layer, Dense, Embedding, LayerNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy

from datasets import load_dataset
from transformers import GPT2Tokenizer

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Data pipeline

In [ ]:
# Configuration
SEQ_LENGTH = 512
BATCH_SIZE = 4

dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = tokenizer.vocab_size
print("Vocab size:", VOCAB_SIZE)
print(dataset)

In [ ]:
def generate_sequences(dataset_split, tokenizer, seq_length):
    """Stream tokens from the split and cut them into (input, target) pairs.

    The target is the input shifted one position left, which is the standard
    next-token-prediction objective.
    """
    buffer = []
    for example in dataset_split:
        text = example["text"]
        if not text.strip():
            continue
        buffer.extend(tokenizer.encode(text))
        while len(buffer) >= seq_length + 1:
            yield (buffer[:seq_length], buffer[1:seq_length + 1])
            buffer = buffer[seq_length:]


def create_tf_dataset(split, shuffle=True, repeat=False):
    generator = lambda: generate_sequences(dataset[split], tokenizer, SEQ_LENGTH)
    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(SEQ_LENGTH,), dtype=tf.int32),
            tf.TensorSpec(shape=(SEQ_LENGTH,), dtype=tf.int32),
        ),
    )
    if repeat:
        ds = ds.repeat()
    if shuffle:
        ds = ds.shuffle(1000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_dataset = create_tf_dataset("train", shuffle=True, repeat=True)
val_dataset = create_tf_dataset("validation", shuffle=False)

# Sanity check on one batch
x, y = next(iter(train_dataset))
print("inputs :", x.shape, x.dtype)
print("targets:", y.shape, y.dtype)
print(tokenizer.decode(x[0][:60].numpy()))

## 3. Model

In [ ]:
class TokenEmbedding(Layer):
    def __init__(self, vocab_size, d_model, **kwargs):
        super().__init__(**kwargs)
        self.embedding = Embedding(vocab_size, d_model)
        self.d_model = d_model

    def call(self, x):
        return self.embedding(x) * tf.math.sqrt(tf.cast(self.d_model, tf.float32))


class PositionalEncoding(Layer):
    def __init__(self, d_model, max_len=5000, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.max_len = max_len

        position = np.arange(max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = np.zeros((max_len, d_model))
        pe[:, 0::2] = np.sin(position * div_term)
        pe[:, 1::2] = np.cos(position * div_term)
        pe = pe[np.newaxis, :, :]

        self.pe = tf.constant(pe, dtype=tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]

In [ ]:
class MultiHeadAttention(Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_model = d_model
        self.depth = d_model // num_heads

        self.wq = Dense(d_model)
        self.wk = Dense(d_model)
        self.wv = Dense(d_model)
        self.dense = Dense(d_model)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    # NOTE: the argument is `look_ahead_mask`, not `mask`. `mask` is reserved by
    # Keras' Layer.__call__ and would be swallowed by the masking machinery.
    def call(self, q, k, v, look_ahead_mask=None):
        batch_size = tf.shape(q)[0]

        q = self.split_heads(self.wq(q), batch_size)
        k = self.split_heads(self.wk(k), batch_size)
        v = self.split_heads(self.wv(v), batch_size)

        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk = tf.cast(tf.shape(k)[-1], tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)

        if look_ahead_mask is not None:
            scaled_attention_logits += (look_ahead_mask * -1e9)

        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)

        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat_output = tf.reshape(output, (batch_size, -1, self.d_model))
        return self.dense(concat_output), attention_weights


class FeedForward(Layer):
    def __init__(self, d_model, dff, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.dense1 = Dense(dff, activation="relu")
        self.dense2 = Dense(d_model)
        self.dropout = Dropout(rate)

    def call(self, x, training=False):
        x = self.dense1(x)
        x = self.dropout(x, training=training)
        return self.dense2(x)


class DecoderLayer(Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, dff, rate)

        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, x, training=False, look_ahead_mask=None):
        attn_output, _ = self.mha(x, x, x, look_ahead_mask=look_ahead_mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1, training=training)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
class Transformer(Model):
    def __init__(self, num_layers, d_model, num_heads, dff, vocab_size, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_layers = num_layers

        self.embedding = TokenEmbedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)

        self.dec_layers = [DecoderLayer(d_model, num_heads, dff, rate)
                           for _ in range(num_layers)]
        self.dropout = Dropout(rate)
        self.final_dense = Dense(vocab_size)

    def call(self, x, training=False, look_ahead_mask=None):
        seq_len = tf.shape(x)[1]

        if look_ahead_mask is None:
            look_ahead_mask = self.create_look_ahead_mask(seq_len)

        x = self.embedding(x)
        x = self.pos_encoding(x)
        x = self.dropout(x, training=training)

        for dec_layer in self.dec_layers:
            # THE FIX: keyword arguments. Passing `training` positionally raises
            # "Only input tensors may be passed as positional arguments."
            x = dec_layer(x, training=training, look_ahead_mask=look_ahead_mask)

        return self.final_dense(x)

    @staticmethod
    def create_look_ahead_mask(size):
        mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
        return mask[tf.newaxis, tf.newaxis, :, :]

## 4. Build and compile

In [ ]:
NUM_LAYERS = 6
D_MODEL = 512
NUM_HEADS = 8
DFF = 2048
DROPOUT_RATE = 0.1

model = Transformer(
    num_layers=NUM_LAYERS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    dff=DFF,
    vocab_size=VOCAB_SIZE,
    rate=DROPOUT_RATE,
)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=SparseCategoricalCrossentropy(from_logits=True),
)

# Force a build so the parameter count is available
_ = model(tf.zeros((1, SEQ_LENGTH), dtype=tf.int32), training=False)
model.summary()

## 5. Train

WikiText-103 is ~1.8M rows, so a full epoch at `BATCH_SIZE = 4` is not something you
finish on a Colab session. `STEPS_PER_EPOCH` and `VALIDATION_STEPS` keep each epoch to a
fixed budget; raise them (or drop them entirely) if you have the compute.

In [ ]:
EPOCHS = 3
STEPS_PER_EPOCH = 500
VALIDATION_STEPS = 50

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_dataset,
    validation_steps=VALIDATION_STEPS,
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], marker="o", label="Training loss")
plt.plot(history.history["val_loss"], marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Training history")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Sample from the model

In [ ]:
def generate(prompt, max_new_tokens=50, temperature=0.8, top_k=50):
    ids = tokenizer.encode(prompt)
    for _ in range(max_new_tokens):
        context = ids[-SEQ_LENGTH:]
        logits = model(tf.constant([context], dtype=tf.int32), training=False)
        logits = logits[0, -1, :] / temperature

        values, indices = tf.math.top_k(logits, k=top_k)
        probs = tf.nn.softmax(values).numpy()
        next_id = int(np.random.choice(indices.numpy(), p=probs / probs.sum()))
        ids.append(next_id)
    return tokenizer.decode(ids)


print(generate("The history of the city", max_new_tokens=40))

Output will be nonsense after only a few hundred steps. A model this size needs on the
order of tens of thousands of steps before the samples become locally coherent.

### Where to go next
- Swap the fixed sinusoidal encoding for learned positional embeddings.
- Use pre-norm (`x + attn(norm(x))`) instead of post-norm; it trains far more stably at depth.
- Add a warmup + cosine decay schedule rather than a flat `1e-4`.
- Tie `final_dense` weights to the token embedding matrix to save ~25M parameters.
- Enable mixed precision: `tf.keras.mixed_precision.set_global_policy("mixed_float16")`.